In [ ]:
import os, json, time, random, shutil
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score,
    precision_recall_fscore_support,
    confusion_matrix, classification_report,
    matthews_corrcoef, cohen_kappa_score
)

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("TF:", tf.__version__)
print("Keras:", keras.__version__)


In [ ]:
WISDM_ROOT = r"E:\WISDM_ar_v1.1"
WISDM_FILE = os.path.join(WISDM_ROOT, "WISDM_ar_v1.1_raw.txt")  # change if your file name differs

RUN_NAME = "WISDM_TRANSFORMER_STREAMLINED"
OUT_DIR = os.path.join(WISDM_ROOT, "OUTPUT_WISDM_TRANSFORMER", RUN_NAME)

DIRS = {
    "conf_mats": os.path.join(OUT_DIR, "confusion_matrices"),
    "models":    os.path.join(OUT_DIR, "models"),
    "plots":     os.path.join(OUT_DIR, "plots"),
    "reports":   os.path.join(OUT_DIR, "reports"),
    "tflite":    os.path.join(OUT_DIR, "tflite"),
}
for d in DIRS.values():
    os.makedirs(d, exist_ok=True)

print("WISDM_FILE:", WISDM_FILE)
print("OUT_DIR:", OUT_DIR)


In [ ]:
def file_size_kb(path):
    return os.path.getsize(path) / 1024

TFLITE_PATHS = {
    "FP32":         os.path.join(DIRS["tflite"], "transformer_fp32.tflite"),
    "FP16":         os.path.join(DIRS["tflite"], "transformer_fp16.tflite"),
    "INT8_DYNAMIC": os.path.join(DIRS["tflite"], "transformer_int8_dynamic.tflite"),
    "INT8_FULL":    os.path.join(DIRS["tflite"], "transformer_int8_full.tflite"),
}

def representative_data():
    idx = np.random.choice(len(X_train_s), size=min(300, len(X_train_s)), replace=False)
    for i in idx:
        yield [X_train_s[i:i+1].astype(np.float32)]

def convert_fp32(savedmodel_dir, out_path):
    c = tf.lite.TFLiteConverter.from_saved_model(savedmodel_dir)
    open(out_path, "wb").write(c.convert())

def convert_fp16(savedmodel_dir, out_path):
    c = tf.lite.TFLiteConverter.from_saved_model(savedmodel_dir)
    c.optimizations = [tf.lite.Optimize.DEFAULT]
    c.target_spec.supported_types = [tf.float16]
    open(out_path, "wb").write(c.convert())

def convert_int8_dynamic(savedmodel_dir, out_path):
    c = tf.lite.TFLiteConverter.from_saved_model(savedmodel_dir)
    c.optimizations = [tf.lite.Optimize.DEFAULT]
    open(out_path, "wb").write(c.convert())

def convert_int8_full(savedmodel_dir, out_path):
    c = tf.lite.TFLiteConverter.from_saved_model(savedmodel_dir)
    c.optimizations = [tf.lite.Optimize.DEFAULT]
    c.representative_dataset = representative_data
    c.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
    c.inference_input_type = tf.int8
    c.inference_output_type = tf.int8
    try:
        open(out_path, "wb").write(c.convert())
        return True
    except Exception as e:
        print("INT8_FULL conversion failed:", str(e)[:350])
        return False

convert_fp32(SAVEDMODEL_DIR, TFLITE_PATHS["FP32"])
convert_fp16(SAVEDMODEL_DIR, TFLITE_PATHS["FP16"])
convert_int8_dynamic(SAVEDMODEL_DIR, TFLITE_PATHS["INT8_DYNAMIC"])
ok_full = convert_int8_full(SAVEDMODEL_DIR, TFLITE_PATHS["INT8_FULL"])
if not ok_full:
    TFLITE_PATHS.pop("INT8_FULL", None)

for k,p in TFLITE_PATHS.items():
    print(k, "->", file_size_kb(p), "KB", "|", p)


In [ ]:
from ai_edge_litert.interpreter import Interpreter

def tflite_predict(tflite_path, X, warmup=40, runs=250):
    interpreter = Interpreter(model_path=tflite_path)
    interpreter.allocate_tensors()

    inp = interpreter.get_input_details()[0]
    out = interpreter.get_output_details()[0]
    in_idx  = inp["index"]
    out_idx = out["index"]

    in_dtype  = inp["dtype"]
    out_dtype = out["dtype"]

    in_scale, in_zero   = inp.get("quantization", (1.0, 0))
    out_scale, out_zero = out.get("quantization", (1.0, 0))

    n = len(X)
    warmup = min(warmup, n)
    timed_runs = min(runs, n)

    probs = np.zeros((n, NUM_CLASSES), dtype=np.float32)

    def quantize_input(x_float):
        q = np.round(x_float / in_scale + in_zero)
        q = np.clip(q, -128, 127).astype(np.int8)
        return q

    def dequantize_output(y):
        return (y.astype(np.float32) - out_zero) * out_scale

    # Warmup
    for i in range(warmup):
        x = X[i:i+1]
        if in_dtype == np.int8:
            x = quantize_input(x)
        interpreter.set_tensor(in_idx, x)
        interpreter.invoke()

    # Timed loop
    t0 = time.perf_counter()
    for i in range(timed_runs):
        x = X[i:i+1]
        if in_dtype == np.int8:
            x = quantize_input(x)

        interpreter.set_tensor(in_idx, x)
        interpreter.invoke()

        y = interpreter.get_tensor(out_idx)
        if out_dtype == np.int8:
            y = dequantize_output(y)

        probs[i] = y[0]

    latency_ms = (time.perf_counter() - t0) / max(timed_runs, 1) * 1000.0

    # Remaining predictions (not timed)
    for i in range(timed_runs, n):
        x = X[i:i+1]
        if in_dtype == np.int8:
            x = quantize_input(x)

        interpreter.set_tensor(in_idx, x)
        interpreter.invoke()

        y = interpreter.get_tensor(out_idx)
        if out_dtype == np.int8:
            y = dequantize_output(y)

        probs[i] = y[0]

    return probs, latency_ms


In [ ]:
def compute_metrics(y_true, probs):
    y_pred = np.argmax(probs, axis=1)

    acc  = accuracy_score(y_true, y_pred)
    bacc = balanced_accuracy_score(y_true, y_pred)

    p_m, r_m, f1_m, _ = precision_recall_fscore_support(y_true, y_pred, average="macro", zero_division=0)
    p_w, r_w, f1_w, _ = precision_recall_fscore_support(y_true, y_pred, average="weighted", zero_division=0)

    mcc   = matthews_corrcoef(y_true, y_pred)
    kappa = cohen_kappa_score(y_true, y_pred)

    return y_pred, dict(
        accuracy=float(acc),
        balanced_accuracy=float(bacc),
        precision_macro=float(p_m),
        recall_macro=float(r_m),
        f1_macro=float(f1_m),
        precision_weighted=float(p_w),
        recall_weighted=float(r_w),
        f1_weighted=float(f1_w),
        mcc=float(mcc),
        kappa=float(kappa),
    )

def keras_latency_ms(model, X, warmup=50, runs=200):
    n = len(X)
    warmup = min(warmup, n)
    runs = min(runs, n)

    for i in range(warmup):
        _ = model.predict(X[i:i+1], verbose=0)
    t0 = time.perf_counter()
    for i in range(runs):
        _ = model.predict(X[i:i+1], verbose=0)
    return (time.perf_counter() - t0) / max(runs, 1) * 1000.0

rows = []

# ---- Keras baseline row (with size+latency) ----
probs_base = best_model.predict(X_test_s, batch_size=256, verbose=0)
y_pred_base, met_base = compute_metrics(y_test, probs_base)

keras_size_kb = os.path.getsize(keras_path) / 1024
lat_keras = keras_latency_ms(best_model, X_test_s)

rows.append({
    "model": "TF_TRANSFORMER_FP32",
    "size_kb": float(keras_size_kb),
    "latency_ms": float(lat_keras),
    **met_base
})

# ---- TFLite variants ----
for name, path in TFLITE_PATHS.items():
    probs, lat = tflite_predict(path, X_test_s)
    y_pred, met = compute_metrics(y_test, probs)

    cm = confusion_matrix(y_test, y_pred)
    plot_cm_values(cm, LABELS, f"TFLITE {name} (WISDM) - Normalized CM",
                   os.path.join(DIRS["conf_mats"], f"TFLITE_TRANSFORMER_{name}_cm_norm.png"),
                   normalize=True)

    rows.append({
        "model": f"TFLITE_TRANSFORMER_{name}",
        "size_kb": float(os.path.getsize(path) / 1024),
        "latency_ms": float(lat),
        **met
    })

final_df = pd.DataFrame(rows)
final_csv = os.path.join(OUT_DIR, "summary_ALL_transformer_variants.csv")
final_df.to_csv(final_csv, index=False)

print("✅ Saved final combined table:", final_csv)
print(final_df.sort_values("f1_macro", ascending=False))


In [ ]:
df = pd.read_csv(os.path.join(OUT_DIR, "summary_ALL_transformer_variants.csv"))

keep = [
    "TFLITE_TRANSFORMER_FP32",
    "TFLITE_TRANSFORMER_FP16",
    "TFLITE_TRANSFORMER_INT8_DYNAMIC",
]
if "TFLITE_TRANSFORMER_INT8_FULL" in df["model"].values:
    keep.append("TFLITE_TRANSFORMER_INT8_FULL")

df = df[df["model"].isin(keep)].copy()
df["model"] = pd.Categorical(df["model"], categories=keep, ordered=True)
df = df.sort_values("model")

pretty = {
    "TFLITE_TRANSFORMER_FP32": "Transformer (FP32)",
    "TFLITE_TRANSFORMER_FP16": "Transformer (FP16)",
    "TFLITE_TRANSFORMER_INT8_DYNAMIC": "Transformer (INT8 Dynamic)",
    "TFLITE_TRANSFORMER_INT8_FULL": "Transformer (INT8 Full)",
}
df["label"] = df["model"].map(pretty)

def hbar(y_labels, values, title, xlabel, save_path, fmt="{:.2f}"):
    y = np.arange(len(y_labels))
    plt.figure(figsize=(10, 4.5))
    bars = plt.barh(y, values)
    plt.yticks(y, y_labels, fontsize=11)
    plt.xlabel(xlabel)
    plt.title(title)
    vmax = np.nanmax(values)
    pad = vmax * 0.01
    for b,v in zip(bars, values):
        plt.text(v + pad, b.get_y()+b.get_height()/2, fmt.format(v), va="center")
    plt.grid(axis="x", alpha=0.3)
    plt.tight_layout()
    plt.savefig(save_path, dpi=600, bbox_inches="tight")
    plt.close()


hbar(df["label"], df["size_kb"].values,
     "WISDM Transformer deployment variants: model size", "Size (KB)",
     os.path.join(DIRS["plots"], "wisdm_transformer_size_kb.png"), fmt="{:.0f}")

hbar(df["label"], (df["accuracy"].values * 100),
     "WISDM Transformer deployment variants: accuracy", "Accuracy (%)",
     os.path.join(DIRS["plots"], "wisdm_transformer_accuracy.png"), fmt="{:.2f}")

hbar(df["label"], (df["f1_macro"].values * 100),
     "WISDM Transformer deployment variants: F1-macro", "F1-macro (%)",
     os.path.join(DIRS["plots"], "wisdm_transformer_f1_macro.png"), fmt="{:.2f}")


In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import label_binarize
from sklearn.metrics import roc_curve, auc


In [ ]:
def roc_multiclass_ovr(y_true, probs, class_names, title, save_path_png, save_path_csv=None):
    """
    Multiclass ROC using One-vs-Rest.
    Saves a single plot with:
      - per-class ROC curves
      - micro-average ROC
      - macro-average ROC
    """
    y_true = np.asarray(y_true).astype(int)
    probs = np.asarray(probs).astype(np.float32)

    n_classes = len(class_names)

    # Binarize labels: shape (N, C)
    y_bin = label_binarize(y_true, classes=np.arange(n_classes))

    # Per-class ROC + AUC
    fpr = {}
    tpr = {}
    roc_auc = {}

    for c in range(n_classes):
        fpr[c], tpr[c], _ = roc_curve(y_bin[:, c], probs[:, c])
        roc_auc[c] = auc(fpr[c], tpr[c])

    # Micro-average
    fpr["micro"], tpr["micro"], _ = roc_curve(y_bin.ravel(), probs.ravel())
    roc_auc["micro"] = auc(fpr["micro"], tpr["micro"])

    # Macro-average (interpolate mean TPR over all unique FPR points)
    all_fpr = np.unique(np.concatenate([fpr[c] for c in range(n_classes)]))
    mean_tpr = np.zeros_like(all_fpr)

    for c in range(n_classes):
        mean_tpr += np.interp(all_fpr, fpr[c], tpr[c])

    mean_tpr /= n_classes
    fpr["macro"], tpr["macro"] = all_fpr, mean_tpr
    roc_auc["macro"] = auc(fpr["macro"], tpr["macro"])

    # ---- Plot ----
    plt.figure(figsize=(9.5, 7.2))

    # Micro + Macro first
    plt.plot(fpr["micro"], tpr["micro"], linewidth=2,
             label=f"micro-average (AUC = {roc_auc['micro']:.4f})")
    plt.plot(fpr["macro"], tpr["macro"], linewidth=2,
             label=f"macro-average (AUC = {roc_auc['macro']:.4f})")

    # Per-class
    for c in range(n_classes):
        plt.plot(fpr[c], tpr[c], linewidth=1.2,
                 label=f"{class_names[c]} (AUC = {roc_auc[c]:.4f})")

    # Chance line
    plt.plot([0, 1], [0, 1], linestyle="--", linewidth=1)

    plt.xlim([0.0, 1.0])
    plt.ylim([0.0, 1.05])
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.title(title)
    plt.grid(alpha=0.25)

    # Put legend outside (clean)
    plt.legend(loc="center left", bbox_to_anchor=(1.02, 0.5), fontsize=9)
    plt.tight_layout()
    plt.savefig(save_path_png, dpi=600, bbox_inches="tight")
    plt.close()

    print("Saved ROC plot:", save_path_png)

    # ---- Save AUC CSV ----
    if save_path_csv is not None:
        rows = [{"class": "micro", "auc": float(roc_auc["micro"])},
                {"class": "macro", "auc": float(roc_auc["macro"])}]
        for c in range(n_classes):
            rows.append({"class": class_names[c], "auc": float(roc_auc[c])})
        pd.DataFrame(rows).to_csv(save_path_csv, index=False)
        print("Saved ROC AUC CSV:", save_path_csv)

    return roc_auc


In [ ]:
# Baseline probs from best_model
probs_tf = best_model.predict(X_test_s, batch_size=256, verbose=0)

roc_png = os.path.join(DIRS["plots"], "WISDM_TF_TRANSFORMER_FP32_ROC.png")
roc_csv = os.path.join(DIRS["plots"], "WISDM_TF_TRANSFORMER_FP32_ROC_AUC.csv")

roc_auc_tf = roc_multiclass_ovr(
    y_true=y_test,
    probs=probs_tf,
    class_names=LABELS,
    title="WISDM TF Transformer FP32 - ROC (OvR)",
    save_path_png=roc_png,
    save_path_csv=roc_csv
)

print("AUC summary:", roc_auc_tf)


In [ ]:
auc_rows = []

for name, path in TFLITE_PATHS.items():
    probs_tfl, _ = tflite_predict(path, X_test_s)  # uses LiteRT version you already added

    roc_png = os.path.join(DIRS["plots"], f"WISDM_TFLITE_TRANSFORMER_{name}_ROC.png")
    roc_csv = os.path.join(DIRS["plots"], f"WISDM_TFLITE_TRANSFORMER_{name}_ROC_AUC.csv")

    roc_auc = roc_multiclass_ovr(
        y_true=y_test,
        probs=probs_tfl,
        class_names=LABELS,
        title=f"WISDM TFLite Transformer {name} - ROC (OvR)",
        save_path_png=roc_png,
        save_path_csv=roc_csv
    )

    auc_rows.append({
        "model": f"TFLITE_TRANSFORMER_{name}",
        "auc_micro": float(roc_auc["micro"]),
        "auc_macro": float(roc_auc["macro"]),
        **{f"auc_{LABELS[i]}": float(roc_auc[i]) for i in range(len(LABELS))}
    })

df_auc = pd.DataFrame(auc_rows)
auc_summary_path = os.path.join(OUT_DIR, "summary_ROC_AUC_all_tflite_variants.csv")
df_auc.to_csv(auc_summary_path, index=False)

print("Saved combined AUC summary:", auc_summary_path)
print(df_auc)


In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import label_binarize
from sklearn.metrics import roc_curve, auc

def micro_macro_roc(y_true, probs, n_classes):
    """Return (fpr_micro, tpr_micro, auc_micro, fpr_macro, tpr_macro, auc_macro)."""
    y_true = np.asarray(y_true).astype(int)
    probs = np.asarray(probs).astype(np.float32)

    y_bin = label_binarize(y_true, classes=np.arange(n_classes))

    # --- Micro ---
    fpr_micro, tpr_micro, _ = roc_curve(y_bin.ravel(), probs.ravel())
    auc_micro = auc(fpr_micro, tpr_micro)

    # --- Macro ---
    fpr = {}
    tpr = {}
    for c in range(n_classes):
        fpr[c], tpr[c], _ = roc_curve(y_bin[:, c], probs[:, c])

    all_fpr = np.unique(np.concatenate([fpr[c] for c in range(n_classes)]))
    mean_tpr = np.zeros_like(all_fpr)

    for c in range(n_classes):
        mean_tpr += np.interp(all_fpr, fpr[c], tpr[c])

    mean_tpr /= n_classes
    fpr_macro, tpr_macro = all_fpr, mean_tpr
    auc_macro = auc(fpr_macro, tpr_macro)

    return fpr_micro, tpr_micro, auc_micro, fpr_macro, tpr_macro, auc_macro


# ---------- Collect curves ----------
n_classes = len(LABELS)

curves = []  # list of dicts: {name, fpr_micro, tpr_micro, auc_micro, fpr_macro, tpr_macro, auc_macro}

# TF FP32
probs_tf = best_model.predict(X_test_s, batch_size=256, verbose=0)
fmi, tmi, aucmi, fma, tma, aucma = micro_macro_roc(y_test, probs_tf, n_classes)
curves.append({
    "name": "TF FP32",
    "fpr_micro": fmi, "tpr_micro": tmi, "auc_micro": aucmi,
    "fpr_macro": fma, "tpr_macro": tma, "auc_macro": aucma,
})

# TFLite variants
for key, path in TFLITE_PATHS.items():
    probs_tfl, _ = tflite_predict(path, X_test_s)
    fmi, tmi, aucmi, fma, tma, aucma = micro_macro_roc(y_test, probs_tfl, n_classes)
    curves.append({
        "name": f"TFLite {key}",
        "fpr_micro": fmi, "tpr_micro": tmi, "auc_micro": aucmi,
        "fpr_macro": fma, "tpr_macro": tma, "auc_macro": aucma,
    })

# ---------- Plot: micro + macro in ONE figure ----------
plt.figure(figsize=(10.5, 7.4))

# Micro curves (solid)
for c in curves:
    plt.plot(c["fpr_micro"], c["tpr_micro"], linewidth=2.0,
             label=f"{c['name']} micro (AUC={c['auc_micro']:.4f})")

# Macro curves (dashed)
for c in curves:
    plt.plot(c["fpr_macro"], c["tpr_macro"], linewidth=2.0, linestyle="--",
             label=f"{c['name']} macro (AUC={c['auc_macro']:.4f})")

# Chance line
plt.plot([0, 1], [0, 1], linestyle=":", linewidth=1.5)

plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("WISDM Transformer: Combined ROC (TF FP32 vs TFLite variants)\nMicro (solid) and Macro (dashed)")
plt.grid(alpha=0.25)

# Legend outside (clean)
plt.legend(loc="center left", bbox_to_anchor=(1.02, 0.5), fontsize=9)
plt.tight_layout()

out_path = os.path.join(DIRS["plots"], "WISDM_ROC_COMBINED_TF_vs_TFLITE_micro_macro.png")
plt.savefig(out_path, dpi=600, bbox_inches="tight")
plt.close()

print("Saved combined ROC:", out_path)

# Print quick summary table
import pandas as pd
summary = pd.DataFrame([{
    "model": c["name"],
    "auc_micro": float(c["auc_micro"]),
    "auc_macro": float(c["auc_macro"]),
} for c in curves]).sort_values("auc_macro", ascending=False)

summary_path = os.path.join(OUT_DIR, "WISDM_ROC_AUC_micro_macro_summary.csv")
summary.to_csv(summary_path, index=False)
print("Saved AUC summary:", summary_path)
print(summary)


In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import label_binarize
from sklearn.metrics import roc_curve, auc
import pandas as pd

def micro_macro_roc(y_true, probs, n_classes):
    y_true = np.asarray(y_true).astype(int)
    probs  = np.asarray(probs).astype(np.float32)

    y_bin = label_binarize(y_true, classes=np.arange(n_classes))

    # Micro-average
    fpr_micro, tpr_micro, _ = roc_curve(y_bin.ravel(), probs.ravel())
    auc_micro = auc(fpr_micro, tpr_micro)

    # Macro-average
    fpr = {}
    tpr = {}
    for c in range(n_classes):
        fpr[c], tpr[c], _ = roc_curve(y_bin[:, c], probs[:, c])

    all_fpr = np.unique(np.concatenate([fpr[c] for c in range(n_classes)]))
    mean_tpr = np.zeros_like(all_fpr)
    for c in range(n_classes):
        mean_tpr += np.interp(all_fpr, fpr[c], tpr[c])
    mean_tpr /= n_classes

    fpr_macro, tpr_macro = all_fpr, mean_tpr
    auc_macro = auc(fpr_macro, tpr_macro)

    return fpr_micro, tpr_micro, auc_micro, fpr_macro, tpr_macro, auc_macro


# -------- Collect curves (TF + all TFLite variants) --------
n_classes = len(LABELS)

curves = []

# TF FP32 baseline
probs_tf = best_model.predict(X_test_s, batch_size=256, verbose=0)
fmi, tmi, aucmi, fma, tma, aucma = micro_macro_roc(y_test, probs_tf, n_classes)
curves.append({
    "name": "TF FP32",
    "fpr_micro": fmi, "tpr_micro": tmi, "auc_micro": aucmi,
    "fpr_macro": fma, "tpr_macro": tma, "auc_macro": aucma,
})

# TFLite variants (LiteRT inference)
for key, path in TFLITE_PATHS.items():
    probs_tfl, _ = tflite_predict(path, X_test_s)
    fmi, tmi, aucmi, fma, tma, aucma = micro_macro_roc(y_test, probs_tfl, n_classes)
    curves.append({
        "name": f"TFLite {key}",
        "fpr_micro": fmi, "tpr_micro": tmi, "auc_micro": aucmi,
        "fpr_macro": fma, "tpr_macro": tma, "auc_macro": aucma,
    })

# -------- Two-panel figure: Micro (left) + Macro (right) --------
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13.2, 5.6))

# --- Left: Micro ---
for c in curves:
    ax1.plot(c["fpr_micro"], c["tpr_micro"], linewidth=2.0,
             label=f"{c['name']} (AUC={c['auc_micro']:.4f})")
ax1.plot([0, 1], [0, 1], linestyle=":", linewidth=1.5)
ax1.set_xlim([0.0, 1.0])
ax1.set_ylim([0.0, 1.05])
ax1.set_xlabel("False Positive Rate")
ax1.set_ylabel("True Positive Rate")
ax1.set_title("Micro-average ROC (OvR)")
ax1.grid(alpha=0.25)

# --- Right: Macro ---
for c in curves:
    ax2.plot(c["fpr_macro"], c["tpr_macro"], linewidth=2.0,
             label=f"{c['name']} (AUC={c['auc_macro']:.4f})")
ax2.plot([0, 1], [0, 1], linestyle=":", linewidth=1.5)
ax2.set_xlim([0.0, 1.0])
ax2.set_ylim([0.0, 1.05])
ax2.set_xlabel("False Positive Rate")
ax2.set_ylabel("True Positive Rate")
ax2.set_title("Macro-average ROC (OvR)")
ax2.grid(alpha=0.25)

# One shared legend to the right of both panels
handles1, labels1 = ax1.get_legend_handles_labels()
fig.legend(handles1, labels1, loc="center left", bbox_to_anchor=(1.02, 0.5), fontsize=9)

fig.suptitle("WISDM Transformer: TF FP32 vs TFLite Variants (ROC)", fontsize=13)
plt.tight_layout(rect=[0, 0, 0.82, 0.95])

out_path = os.path.join(DIRS["plots"], "WISDM_ROC_TWOPANEL_micro_macro.png")
plt.savefig(out_path, dpi=600, bbox_inches="tight")
plt.close()

print("Saved two-panel ROC:", out_path)

# -------- Save compact AUC summary (micro + macro only) --------
summary = pd.DataFrame([{
    "model": c["name"],
    "auc_micro": float(c["auc_micro"]),
    "auc_macro": float(c["auc_macro"]),
} for c in curves]).sort_values("auc_macro", ascending=False)

summary_path = os.path.join(OUT_DIR, "WISDM_ROC_AUC_micro_macro_summary.csv")
summary.to_csv(summary_path, index=False)

print("Saved AUC summary:", summary_path)
print(summary)

In [ ]:
import os
import pandas as pd
import numpy as np

# ---- Load WISDM Transformer summary ----
summary_path = os.path.join(OUT_DIR, "summary_ALL_transformer_variants.csv")
df = pd.read_csv(summary_path)

print("Loaded summary shape:", df.shape)

# Ensure numeric
for c in ["size_kb", "latency_ms"]:
    df[c] = pd.to_numeric(df[c], errors="coerce")

# =========================
# 1) FPS (frames per second)
# =========================
df["fps"] = 1000.0 / df["latency_ms"]
df["fps"] = df["fps"].replace([np.inf, -np.inf], np.nan)

# =========================
# 2) Compression ratio
# =========================

# Reference FP32 Keras model
fp32_row = df[df["model"] == "TF_TRANSFORMER_FP32"]

if fp32_row.empty:
    raise ValueError(" TF_TRANSFORMER_FP32 row not found in WISDM summary!")

fp32_size = float(fp32_row["size_kb"].values[0])
print("FP32 baseline size (KB):", fp32_size)

# Compression ratio = FP32_size / model_size
df["compression_ratio"] = fp32_size / df["size_kb"]

# For baseline itself
df.loc[df["model"] == "TF_TRANSFORMER_FP32", "compression_ratio"] = 1.0

# =========================
# Save updated table
# =========================
out_path = os.path.join(OUT_DIR, "summary_ALL_transformer_variants_with_FPS_CR.csv")
df.to_csv(out_path, index=False)

print("Saved WISDM summary with FPS + Compression ratio:")
print(out_path)

# =========================
# Show compact view
# =========================
cols_show = [
    "model",
    "accuracy",
    "f1_macro",
    "size_kb",
    "latency_ms",
    "fps",
    "compression_ratio"
]

print(df[cols_show].sort_values("compression_ratio", ascending=False))


In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ---- Load summary ----
summary_path = os.path.join(OUT_DIR, "summary_ALL_transformer_variants.csv")
df = pd.read_csv(summary_path)

# Force to plain string (prevents categorical category mismatch errors)
df["model"] = df["model"].astype(str)

# ---- Keep/order models ----
keep_order = [
    "TF_TRANSFORMER_FP32",
    "TFLITE_TRANSFORMER_FP32",
    "TFLITE_TRANSFORMER_FP16",
    "TFLITE_TRANSFORMER_INT8_DYNAMIC",
    "TFLITE_TRANSFORMER_INT8_FULL",
]
keep_order = [m for m in keep_order if m in df["model"].values]  # only those present

df = df[df["model"].isin(keep_order)].copy()

# Safe categorical assignment (now model is string)
df["model"] = pd.Categorical(df["model"], categories=keep_order, ordered=True)
df = df.sort_values("model")

# Pretty names
pretty = {
    "TF_TRANSFORMER_FP32": "TF FP32",
    "TFLITE_TRANSFORMER_FP32": "TFLite FP32",
    "TFLITE_TRANSFORMER_FP16": "TFLite FP16",
    "TFLITE_TRANSFORMER_INT8_DYNAMIC": "TFLite INT8-Dyn",
    "TFLITE_TRANSFORMER_INT8_FULL": "TFLite INT8-Full",
}
df["label"] = df["model"].astype(str).map(pretty).fillna(df["model"].astype(str))

# Ensure numeric
for c in ["accuracy", "f1_macro", "size_kb", "latency_ms"]:
    df[c] = pd.to_numeric(df[c], errors="coerce")

# ---- Normalize metrics for a single-axis plot ----
def minmax_norm(arr):
    arr = np.asarray(arr, dtype=np.float64)
    mn = np.nanmin(arr)
    mx = np.nanmax(arr)
    if not np.isfinite(mn) or not np.isfinite(mx) or mx == mn:
        return np.zeros_like(arr)
    return (arr - mn) / (mx - mn)

acc_n  = minmax_norm(df["accuracy"].values)
f1_n   = minmax_norm(df["f1_macro"].values)
size_n = minmax_norm(df["size_kb"].values)
lat_n  = minmax_norm(df["latency_ms"].values)

labels = df["label"].tolist()
x = np.arange(len(labels))
bar_w = 0.20

fig, ax = plt.subplots(figsize=(13.5, 5.8))

b1 = ax.bar(x - 1.5*bar_w, acc_n,  width=bar_w, label="Accuracy (norm)")
b2 = ax.bar(x - 0.5*bar_w, f1_n,   width=bar_w, label="F1-macro (norm)")
b3 = ax.bar(x + 0.5*bar_w, size_n, width=bar_w, label="Model size KB (norm)")
b4 = ax.bar(x + 1.5*bar_w, lat_n,  width=bar_w, label="Latency ms (norm)")

ax.set_xticks(x)
ax.set_xticklabels(labels, rotation=20, ha="right", fontsize=11)
ax.set_ylim(0, 1.15)
ax.set_ylabel("Normalized value (0–1)", fontsize=12)
ax.set_title("WISDM Transformer Variants: Accuracy, F1, Model Size, Latency (single view)", fontsize=13)
ax.grid(axis="y", alpha=0.25)
ax.legend(loc="upper left", fontsize=10)

# ---- Annotate bars with REAL values ----
def annotate(bars, real_vals, fmt):
    for bar, v in zip(bars, real_vals):
        if not np.isfinite(v):
            continue
        ax.text(
            bar.get_x() + bar.get_width()/2,
            bar.get_height() + 0.02,
            fmt.format(v),
            ha="center", va="bottom",
            fontsize=9, rotation=90
        )

annotate(b1, df["accuracy"].values * 100.0, "{:.2f}%")
annotate(b2, df["f1_macro"].values * 100.0, "{:.2f}%")
annotate(b3, df["size_kb"].values, "{:.0f}KB")
annotate(b4, df["latency_ms"].values, "{:.2f}ms")

plt.tight_layout()
out_path = os.path.join(DIRS["plots"], "WISDM_single_bar_acc_f1_size_latency.png")
plt.savefig(out_path, dpi=600, bbox_inches="tight")
plt.close()

